In [1]:
# === Setup & Imports ===
import sys
import joblib
import json
from pathlib import Path
project_root = Path().resolve().parent  
sys.path.insert(0, str(project_root))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from IPython.display import Image, display
from sklearn.preprocessing import FunctionTransformer

from creditcard_psp.features import add_time_features, compute_feature_cols

from creditcard_psp.modeling.train import (
    load_fees, compute_feature_cols, make_preprocessor_selector, get_models,
    metrics, plot_calibration, predict_p_mat,
    shap_summary_plot, _to_display, save_dashboard_artifacts,
    RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, RNG
)

df_path = PROCESSED_DATA_DIR / "df.pkl"
fee_xlsx = RAW_DATA_DIR / "PSP_Servicegebuehren.xlsx"
calib = "sigmoid"
cv = 3
test_size = 0.2
selection_metric = "log_loss"
# models = ["LOGREG","RF","XGB"]
models = ["LOGREG","XGB"]
dump_dashboard_artifacts = True
do_shap = True

2025-10-14 16:31:35.400 | INFO     | creditcard_psp.config:<module>:12 - PROJ_ROOT path is: C:\Users\miria\creditcard_psp


In [2]:
"""Trainiert, evaluiert und speichert die PSP-Routing-Modelle."""
print("--- Starte Trainings-Pipeline ---")

# ---- Load df (Parquet bevorzugt, mit Fallback) ----
try:
    if str(df_path).lower().endswith(".parquet"):
        df = pd.read_parquet(df_path)
    elif str(df_path).lower().endswith((".pkl", ".pickle", ".joblib")):
        try:
            df = pd.read_pickle(df_path)
        except Exception:
            df = joblib.load(df_path)
    else:
        try:
            df = pd.read_parquet(df_path)
        except Exception:
            try:
                df = pd.read_pickle(df_path)
            except Exception:
                df = joblib.load(df_path)
except Exception as e:
    raise RuntimeError(f"DF konnte nicht geladen werden ({df_path}): {e}")

target, feature_cols = compute_feature_cols(df)
X_tr, X_te, y_tr, y_te = train_test_split(
    df[feature_cols], df[target].astype(int),
    test_size=test_size, random_state=RNG, stratify=df[target].astype(int)
)
fees = load_fees(fee_xlsx)
psps = sorted(list(X_tr["PSP"].dropna().unique()))

feat_step = ("feat", FunctionTransformer(add_time_features, validate=False))
pre = make_preprocessor_selector()
zoo = get_models()

# --- Train & Evaluate Loop ---
results_data = []
trained_models = {}
for name in models:
    print(f"\n--- Training Modell: {name} ---")
    base = zoo[name]

    # Kalibriertes Modell
    pipe = Pipeline([feat_step, ("preprocessor", pre), ("classifier", base)])
    clf = CalibratedClassifierCV(estimator=pipe, cv=cv, method=calib).fit(X_tr, y_tr)

    # Metriken
    y_proba = clf.predict_proba(X_te)[:, 1]
    m = metrics(y_te, y_proba)
    results_data.append({"model": name, **m})
    plot_calibration(y_te, y_proba, FIGURES_DIR / f"calibration_{name}.png", f"Calibration - {name}")

    # Unkalibriertes Modell (für SHAP)
    pipe_raw = Pipeline([feat_step, ("preprocessor", pre), ("classifier", base)]).fit(X_tr, y_tr)
    trained_models[name] = {"calibrated": clf, "raw": pipe_raw}

    if do_shap:
        shap_summary_plot(pipe_raw, X_te, name, FIGURES_DIR / f"shap_{name}.png")

# --- Bestes Modell wählen ---
summary_df = pd.DataFrame(results_data).sort_values(by=selection_metric)
best_name = summary_df.iloc[0]["model"]
best_model_calibrated = trained_models[best_name]["calibrated"]
best_model_raw = trained_models[best_name]["raw"]

print(f"\n--- Bestes Modell (nach {selection_metric}): {best_name} ---")
print(summary_df)

# ---------- Mapping/Display-Namen erzeugen ----------
pre_best = best_model_raw.named_steps["preprocessor"]
try:
    feat_out = pre_best.get_feature_names_out()
except Exception:
    # Fallback, falls get_feature_names_out nicht verfügbar ist
    X_tmp = pre_best.transform(X_tr.iloc[:1])
    feat_out = np.array([f"f_{i}" for i in range(X_tmp.shape[1])])

feat_out = list(map(str, feat_out))
display_names = [_to_display(n) for n in feat_out]
mapping = dict(zip(feat_out, display_names))

# --- Artefakte (Modelle/Meta) im MODELS_DIR speichern ---
summary_df.to_csv(MODELS_DIR / "summary_metrics.csv", index=False)
joblib.dump(best_model_calibrated, MODELS_DIR / f"best_model_{best_name}.joblib")
with open(MODELS_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_model_name": best_name,
        "feature_cols": feature_cols,
        "psps": psps,
        "feature_names_out": feat_out,
        "feature_names_display": display_names,
        "mapping": mapping
    }, f, indent=2)

# --- Dashboard-Artefakte ---
if dump_dashboard_artifacts:
    print("\n--- Speichere Artefakte für Dashboard ---")
    DASHBOARD_DATA_DIR = Path("dashboard_data")
    DASHBOARD_DATA_DIR.mkdir(exist_ok=True)

    # Modelle: joblib
    joblib.dump(best_model_calibrated, DASHBOARD_DATA_DIR / "psp_router_model.joblib")
    joblib.dump(best_model_raw,        DASHBOARD_DATA_DIR / "model_raw.joblib")

    # Original-DF: Parquet
    df.to_parquet(DASHBOARD_DATA_DIR / "df.parquet", index=True)

    # Test-Features: Parquet
    X_te.to_parquet(DASHBOARD_DATA_DIR / "X_te.parquet", index=True)

    # PSPs
    with open(DASHBOARD_DATA_DIR / "psps.json", 'w', encoding="utf-8") as f:
        json.dump(psps, f)

    # Matrizen
    P = predict_p_mat(best_model_calibrated, X_te[feature_cols], psps, "PSP")
    np.save(DASHBOARD_DATA_DIR / "P_matrix.npy", P)

    fee_s = np.array([fees.get(p, {}).get("fee_successful", 0.0) for p in psps], dtype=float)
    fee_f = np.array([fees.get(p, {}).get("fee_not_successful", 0.0) for p in psps], dtype=float)
    exp_cost = P * fee_s + (1 - P) * fee_f
    np.save(DASHBOARD_DATA_DIR / "exp_cost_matrix.npy", exp_cost)

    # Meta (Mapping) auch fürs Dashboard
    with open(DASHBOARD_DATA_DIR / "meta.json", "w", encoding="utf-8") as f:
        json.dump({
            "feature_names_out": feat_out,
            "feature_names_display": display_names,
            "mapping": mapping
        }, f, indent=2)

    print("Alle Dashboard-Artefakte gespeichert.")

--- Starte Trainings-Pipeline ---

--- Training Modell: LOGREG ---
SHAP gespeichert: C:\Users\miria\creditcard_psp\reports\figures\shap_LOGREG.png

--- Training Modell: XGB ---
SHAP gespeichert: C:\Users\miria\creditcard_psp\reports\figures\shap_XGB.png

--- Bestes Modell (nach log_loss): XGB ---
    model   auc_roc  avg_precision  log_loss     brier  accuracy
1     XGB  0.664980       0.343779   0.47843  0.152965  0.799339
0  LOGREG  0.636261       0.313769   0.48746  0.156210  0.797753

--- Speichere Artefakte für Dashboard ---
Alle Dashboard-Artefakte gespeichert.
